# TweetAge v7: ultima ronda de hiperparametros de n-gramas

Esta es la **ultima ronda de v7 sobre la familia de n-gramas** y se cierra despues de esta corrida. Se mantiene la normalizacion A de v6, el ModifiedKN por clase sin cambios, score bruto mas log prior y ausencia de calibracion de sesgos. Se reutiliza la particion v7_lower en minusculas, verificando SHA-256 del NPZ, del train y de los grupos antes de entrenar.

Ocho configuraciones fijadas, en este orden de desempate:
1. caracteres 3, palabras 2, lambda 0.6;
2. caracteres 3, palabras 2, lambda 0.9;
3. caracteres 3, palabras 4, lambda 0.6;
4. caracteres 3, palabras 4, lambda 0.9;
5. caracteres 5, palabras 2, lambda 0.6;
6. caracteres 5, palabras 2, lambda 0.9;
7. caracteres 5, palabras 4, lambda 0.6;
8. caracteres 5, palabras 4, lambda 0.9.

La referencia (4,3,0.75) se reentrena y se recalcula en los mismos cinco pliegues; el valor anterior 0.440081 se muestra solo como contexto, nunca se utiliza para la seleccion.

Regla fijada: una configuracion se acepta si la media de las cinco diferencias pareadas de F1 macro contra la referencia recalculada es positiva y mayor que su desviacion estandar muestral (ddof=1). Se elige la mayor media entre aceptadas; en empate, el orden de la lista. Si ninguna se acepta, se omiten holdout y CSV. Esta regla es una heuristica, no una prueba formal de significacion. Elegir entre ocho configuraciones introduce optimismo de seleccion.

Se reutilizan los conteos entre ordenes dentro de un mismo ajuste para ahorrar tiempo: se comprueba que esto equivale a entrenarlos por separado. No se comparten vocabularios, conteos ni priors entre pliegues. El holdout no se utiliza para elegir y, si procede, se evalua una sola vez. No se regeneran particiones ni se modifican archivos existentes; no se hacen commit, push ni envios a Kaggle.

In [1]:
import ast
import csv
import gc
import hashlib
import html
import json
import math
import platform
import re
import time
from collections import Counter
from functools import lru_cache
from itertools import product
from pathlib import Path
import numpy as np
import pandas as pd
import sklearn
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from IPython.display import display, Markdown

SEED=42
np.random.seed(SEED)
CLASSES=np.array(['13-17','18-24','25-34','35-49','50-64','65-xx'])
BASE_EPSILON=1e-6
BOS,EOS,UNK='\x02','\x03','\x01'
CONFIGURATIONS=[dict(id=f'C{index}',orden_lista=index,car=char_order,pal=word_order,peso=weight) for index,(char_order,word_order,weight) in enumerate(product([3,5],[2,4],[0.6,0.9]),1)]
REFERENCE=dict(id='REF',orden_lista=0,car=4,pal=3,peso=0.75)
ALL_CONFIGURATIONS=[REFERENCE]+CONFIGURATIONS
CHAR_ORDERS=[3,4,5]
WORD_ORDERS=[2,3,4]
roots=[Path.cwd(),Path.cwd()/'workshops'/'Kaggle-tweet-age',Path.cwd()/'natural-language-processing-workshops'/'workshops'/'Kaggle-tweet-age']
OUT_DIR=next((candidate for candidate in roots if (candidate/'TweetAge_v7_hiperparametros.ipynb').is_file()),None)
assert OUT_DIR is not None,'Ejecute desde la carpeta del notebook o la raiz del repo.'
NOTEBOOK_PATH=OUT_DIR/'TweetAge_v7_hiperparametros.ipynb'
SUB_PATH=OUT_DIR/'age_submission_v7_hiperparametros.csv'
SPLIT_PATH=OUT_DIR/'split_idx_v7_lower_seed42.npz'
DATA_DIR=OUT_DIR/'tweet-age-2026-puj'
assert not SUB_PATH.exists(),'El CSV ya existe; no se sobrescribe.'
INPUT_HASHES={name:hashlib.sha256((DATA_DIR/name).read_bytes()).hexdigest() for name in ['age_train.csv','age_test.csv','age_submission.csv']}
PROTECTED_HASHES={path.name:hashlib.sha256(path.read_bytes()).hexdigest() for path in OUT_DIR.iterdir() if path.is_file() and path!=NOTEBOOK_PATH}
assert PROTECTED_HASHES[SPLIT_PATH.name]=='692fcfd2fc5a8e1ead91763d4741da5d14f86fecd201b2e419c5e7d47c16be83','Hash del NPZ v7_lower distinto; detener.'
print('Python',platform.python_version(),'numpy',np.__version__,'pandas',pd.__version__,'sklearn',sklearn.__version__)
print('Semilla',SEED,'notebook',NOTEBOOK_PATH.name,'split',SPLIT_PATH.name)
print('SHA-256 NPZ v7_lower:',PROTECTED_HASHES[SPLIT_PATH.name])
def check_compliance(notebook):
    forbidden = {'tfidfvectorizer', 'countvectorizer', 'hashingvectorizer', 'dictvectorizer', 'feature_extraction', 'linear_model', 'transformers', 'transformer', 'embedding', 'embeddings', 'word2vec', 'glove'}
    problems = []
    for cell_number, cell in enumerate(notebook['cells'], 1):
        if cell['cell_type'] != 'code':
            continue
        source = ''.join(cell['source']) if isinstance(cell['source'], list) else cell['source']
        for node in ast.walk(ast.parse(source)):
            names = []
            if isinstance(node, ast.Name):
                names = [node.id]
            elif isinstance(node, ast.Attribute):
                names = [node.attr]
            elif isinstance(node, ast.Import):
                names = [alias.name for alias in node.names]
            elif isinstance(node, ast.ImportFrom):
                names = [node.module or ''] + [alias.name for alias in node.names]
            for name in names:
                if any(part.lower() in forbidden for part in name.split('.')):
                    problems.append((cell_number, node.lineno, name))
            if isinstance(node, ast.Call) and isinstance(node.func, ast.Name) and node.func.id in {'eval', 'exec', '__import__'}:
                problems.append((cell_number, node.lineno, 'ejecucion dinamica'))
    if problems:
        raise RuntimeError(f'Guardia AST: codigo prohibido {problems}')
    return True

with NOTEBOOK_PATH.open(encoding='utf-8') as handle:
    assert check_compliance(json.load(handle))
print('Guardia AST de v6 superada antes de entrenar.')
display(pd.DataFrame(ALL_CONFIGURATIONS))


Python 3.12.14 numpy 2.5.4 pandas 3.0.6 sklearn 1.9.1
Semilla 42 notebook TweetAge_v7_hiperparametros.ipynb split split_idx_v7_lower_seed42.npz
SHA-256 NPZ v7_lower: 692fcfd2fc5a8e1ead91763d4741da5d14f86fecd201b2e419c5e7d47c16be83
Guardia AST de v6 superada antes de entrenar.


,id,orden_lista,car,pal,peso
0,REF,0,4,3,0.75
1,C1,1,3,2,0.60
2,C2,2,3,2,0.90
3,C3,3,3,4,0.60
4,C4,4,3,4,0.90
5,C5,5,5,2,0.60
6,C6,6,5,2,0.90
7,C7,7,5,4,0.60
8,C8,8,5,4,0.90


In [2]:
def verify_csv(path, columns, delimiter, bom):
    with path.open('rb') as handle:
        prefix = handle.read(32768)
    first_line = prefix.decode('utf-8-sig').splitlines()[0]
    actual = csv.Sniffer().sniff(first_line, delimiters=',;\t|').delimiter
    header = next(csv.reader([first_line], delimiter=actual))
    assert actual == delimiter and prefix.startswith(b'\xef\xbb\xbf') == bom, (path.name, actual)
    assert header == columns, (path.name, header)
    print(path.name, 'BOM=', bom, 'separador=', repr(actual), 'columnas=', header)
    return header

URL_RE = re.compile(r'https?://[^\s]+|www\.[^\s]+', re.IGNORECASE)

MENTION_RE = re.compile(r'@\w+', re.UNICODE)

REPEAT_RE = re.compile(r'([^\W\d_])\1{3,}', re.UNICODE)

def replace_url(match):
    value = match.group(0)
    core = value.rstrip('.,!?;:)]}')
    return '<url>' + value[len(core):]

def normalize_text(text):
    return REPEAT_RE.sub(lambda match: match.group(1) * 3, MENTION_RE.sub('<user>', URL_RE.sub(replace_url, text)))

verify_csv(DATA_DIR/'age_train.csv',['id','text','age_range'],';',True)
reference_columns=verify_csv(DATA_DIR/'age_submission.csv',['id','age_range'],',',False)
train_raw=pd.read_csv(DATA_DIR/'age_train.csv',sep=';',encoding='utf-8-sig',dtype=str,keep_default_na=False)
assert list(train_raw.columns)==['id','text','age_range']
clean=train_raw.text.fillna('').str.strip().map(html.unescape)
excluded=clean.eq('')|clean.str.contains('#\u00bfNOMBRE',regex=False)
train=train_raw.loc[~excluded].copy()
train['raw_row']=np.flatnonzero(~excluded.to_numpy())
train['text']=clean.loc[~excluded]
train=train.reset_index(drop=True)
assert train.id.is_unique and set(train.age_range)==set(CLASSES)
train['normalized']=train.text.map(normalize_text)
y_all=pd.Categorical(train.age_range,categories=CLASSES).codes.astype(np.int64)
with np.load(SPLIT_PATH,allow_pickle=False) as saved:
    split={name:saved[name].copy() for name in saved.files}
assert str(split['train_sha256'])==INPUT_HASHES['age_train.csv'] and int(split['seed'])==SEED
assert np.array_equal(split['retained_raw_idx'],train.raw_row.to_numpy())
group_keys=np.array([text.lower() for text in train.normalized],dtype=object)
groups_hash=hashlib.sha256(json.dumps(group_keys.tolist(),ensure_ascii=False,separators=(',',':')).encode('utf-8')).hexdigest()
assert groups_hash==str(split['groups_sha256'])
group_ids,unique_keys=pd.factorize(group_keys,sort=True)
assert np.array_equal(group_ids,split['group_ids'])
idx_development,idx_holdout=split['train_idx'],split['valid_idx']
dev_groups,holdout_groups=split['train_groups'],split['valid_groups']
assert np.array_equal(np.sort(np.concatenate([idx_development,idx_holdout])),np.arange(len(train)))
assert np.array_equal(train.raw_row.to_numpy()[idx_development],split['raw_train_idx'])
assert np.array_equal(train.raw_row.to_numpy()[idx_holdout],split['raw_valid_idx'])
assert set(group_ids[idx_development])==set(dev_groups) and set(group_ids[idx_holdout])==set(holdout_groups)
assert not np.intersect1d(dev_groups,holdout_groups).size
assert not set(group_keys[idx_development])&set(group_keys[idx_holdout])
group_counts=np.zeros((len(unique_keys),6),dtype=np.int64)
np.add.at(group_counts,(group_ids,y_all),1)
group_targets=group_counts.argmax(axis=1)
texts_development=train.normalized.iloc[idx_development].to_numpy()
y_development=y_all[idx_development]
groups_development=group_ids[idx_development]
group_folds=list(StratifiedKFold(n_splits=5,shuffle=True,random_state=SEED).split(dev_groups,group_targets[dev_groups]))
folds=[]
leak_rows=[dict(validacion='holdout',textos_compartidos=0)]
for fold_number,(fit_positions,app_positions) in enumerate(group_folds,1):
    fit_idx=np.flatnonzero(np.isin(groups_development,dev_groups[fit_positions]))
    app_idx=np.flatnonzero(np.isin(groups_development,dev_groups[app_positions]))
    shared={text.lower() for text in texts_development[fit_idx]}&{text.lower() for text in texts_development[app_idx]}
    leak_rows.append(dict(validacion=str(fold_number),textos_compartidos=len(shared)))
    if shared:
        print('DETENER: fuga en pliegue',fold_number,'ejemplos',sorted(shared)[:10])
        raise RuntimeError('Fuga en minusculas; no entrenar.')
    folds.append((fit_idx,app_idx))
print('NPZ REUTILIZADO SIN REGENERAR:',SPLIT_PATH.name)
print('SHA-256 train y grupos correctos. SHA-256 grupos:',groups_hash)
print('Filas train original/limpio/excluidas:',len(train_raw),len(train),int(excluded.sum()))
print('Desarrollo grupos/filas:',len(dev_groups),len(idx_development),'holdout grupos/filas:',len(holdout_groups),len(idx_holdout))
print('Textos solo URL:',int(train.normalized.str.strip().eq('<url>').sum()),'solo usuario:',int(train.normalized.str.strip().eq('<user>').sum()))
display(pd.DataFrame(leak_rows))


age_train.csv BOM= True separador= ';' columnas= ['id', 'text', 'age_range']
age_submission.csv BOM= False separador= ',' columnas= ['id', 'age_range']


NPZ REUTILIZADO SIN REGENERAR: split_idx_v7_lower_seed42.npz
SHA-256 train y grupos correctos. SHA-256 grupos: f482fe9f4a7495a0050b857b8eda3b3bc9177c31e2ae6f10c9138cfe8df2b1a8
Filas train original/limpio/excluidas: 169336 169181 155
Desarrollo grupos/filas: 126309 132987 holdout grupos/filas: 31184 36194
Textos solo URL: 3785 solo usuario: 287


,validacion,textos_compartidos
0,holdout,0
1,1,0
2,2,0
3,3,0
4,4,0
5,5,0


In [3]:
TOKEN_RE = re.compile(r'<url>|<user>|\w+|[^\w\s]', re.UNICODE)

def tokenize(text, level):
    return list(text) if level == 'char' else TOKEN_RE.findall(text.lower())

def make_encoder(texts, level):
    frequencies = Counter()
    for text in texts:
        frequencies.update(tokenize(text, level))
    cutoff = 1 if level == 'char' else 2
    return {token: chr(index + 256) for index, token in enumerate(sorted(token for token, count in frequencies.items() if count >= cutoff))}

def encode_text(text, level, mapping):
    return ''.join(mapping.get(token, UNK) for token in tokenize(text, level))

def estimate_discounts(counts):
    histogram = Counter(counts.values())
    count1, count2, count3, count4 = [histogram.get(value, 0) for value in [1, 2, 3, 4]]
    factor = count1 / (count1 + 2 * count2) if count1 + 2 * count2 else 0.0
    first = 1 - 2 * factor * count2 / count1 if count1 and count2 else 0.75
    second = 2 - 3 * factor * count3 / count2 if count1 and count2 else 0.75
    third = 3 - 4 * factor * count4 / count3 if count1 and count2 and count3 else 0.75
    values = tuple(float(min(bucket - 1e-6, max(1e-6, value))) for bucket, value in enumerate([first, second, third], 1))
    return values, dict(N1=count1, N2=count2, N3=count3, N4=count4)

def discount_for(count, discounts):
    return discounts[min(count, 3) - 1] if count > 0 else 0.0

class ModifiedKN:
    def __init__(self, max_order, support):
        self.max_order = max_order
        self.support = set(support) | {EOS, UNK}

    def fit(self, sequences):
        highest = max(2, self.max_order)
        self.raw = {order: Counter() for order in range(2, highest + 1)}
        for sequence in sequences:
            padded = BOS * (highest - 1) + sequence + EOS
            for order, counts in self.raw.items():
                counts.update(padded[position - order + 1:position + 1] for position in range(highest - 1, len(padded)))
        self.continuation = {}
        for order in range(1, highest):
            effective = Counter()
            for gram in self.raw[order + 1]:
                effective[gram[1:]] += 1
            self.continuation[order] = effective
        self.base = self.continuation[1]
        self.base_total = sum(self.base.values()) + BASE_EPSILON * len(self.support)
        self.discount_records = []
        def prepare(count_tables, origin):
            discounts_by_order, histories_by_order = {}, {}
            for order, counts in count_tables.items():
                if order == 1:
                    continue
                discounts, histogram = estimate_discounts(counts)
                discounts_by_order[order] = discounts
                totals, removed = Counter(), Counter()
                for gram, count in counts.items():
                    totals[gram[:-1]] += count
                    removed[gram[:-1]] += discount_for(count, discounts)
                histories_by_order[order] = {history: (total, removed[history]) for history, total in totals.items()}
                self.discount_records.append(dict(origen=origin, orden=order, D1=discounts[0], D2=discounts[1], D3=discounts[2], **histogram))
            return discounts_by_order, histories_by_order
        self.raw_discounts, self.raw_hist = prepare(self.raw, 'frecuencias')
        self.cont_discounts, self.cont_hist = prepare(self.continuation, 'continuaciones')
        self.lower_cached = lru_cache(maxsize=200000)(self._lower)
        self.log_cached = lru_cache(maxsize=200000)(self._log_gram)
        return self

    def _lower(self, gram):
        if len(gram) == 1:
            return (self.base.get(gram, 0) + BASE_EPSILON) / self.base_total
        order = len(gram)
        lower = self.lower_cached(gram[1:])
        total, removed = self.cont_hist[order].get(gram[:-1], (0, 0.0))
        if not total:
            return lower
        count = self.continuation[order].get(gram, 0)
        return (count - discount_for(count, self.cont_discounts[order]) + removed * lower) / total

    def probability(self, gram):
        if len(gram) == 1:
            return self.lower_cached(gram)
        order = len(gram)
        lower = self.lower_cached(gram[1:])
        total, removed = self.raw_hist[order].get(gram[:-1], (0, 0.0))
        if not total:
            return lower
        count = self.raw[order].get(gram, 0)
        return (count - discount_for(count, self.raw_discounts[order]) + removed * lower) / total

    def _log_gram(self, gram):
        probability = self.probability(gram)
        if not 0 < probability <= 1 + 1e-10:
            raise ArithmeticError((repr(gram), probability))
        return math.log(probability)

    def log_probability(self, sequence, order):
        assert 1 <= order <= self.max_order
        padded = BOS * (order - 1) + sequence + EOS
        grams = Counter(padded[position - order + 1:position + 1] for position in range(order - 1, len(padded)))
        return sum(count * self.log_cached(gram) for gram, count in grams.items())

    def close(self):
        self.lower_cached.cache_clear()
        self.log_cached.cache_clear()

def test_modified_kn():
    values, _ = estimate_discounts(Counter({'a': 1, 'b': 2, 'c': 3, 'd': 4, 'e': 1, 'f': 2, 'g': 3, 'h': 1}))
    assert len(set(values)) == 3 and all(0 < value < bucket for bucket, value in enumerate(values, 1))
    model = ModifiedKN(4, set('abc')).fit(['ab', 'ab', 'ac', 'bc', 'abc', 'abc', 'abc'])
    assert model.base['a'] == 1 and model.raw[2][BOS + 'a'] == 6
    for order in [1, 2, 3, 4]:
        contexts = set(model.raw_hist.get(order, {})) | {'', 'zz', BOS * (order - 1)}
        for context in contexts:
            history = (BOS * (order - 1) + context)[-(order - 1):] if order > 1 else ''
            assert abs(sum(model.probability(history + symbol) for symbol in model.support) - 1) < 1e-9, (order, repr(history))
        assert math.isfinite(model.log_probability('a' + UNK, order))
    assert model.log_probability('ab', 4) > model.log_probability('cc', 4)
    model.close()
    assert tokenize('Hola HOLA', 'word') == ['hola', 'hola']
    assert tokenize('Hola', 'char')[0] == 'H'
    print('Pruebas MKN: tres descuentos, normalizacion, continuaciones, BOS/EOS, UNK y capitalizacion correctos.')

def test_shared_counts():
    toy_texts=['Hola hola!','Hola amiga!','Adios hola!','Hola hola!','Amiga adios!']
    for level,orders in [('char',CHAR_ORDERS),('word',WORD_ORDERS)]:
        mapping=make_encoder(toy_texts,level)
        sequences=[encode_text(text,level,mapping) for text in toy_texts]
        support=set(mapping.values())
        shared_model=ModifiedKN(max(orders),support).fit(sequences)
        for order in orders:
            separate_model=ModifiedKN(order,support).fit(sequences)
            for sequence in sequences+[UNK]:
                assert abs(shared_model.log_probability(sequence,order)-separate_model.log_probability(sequence,order))<1e-12,(level,order)
            separate_model.close()
        shared_model.close()
    print('Prueba de conteos compartidos: mismos log P que entrenar cada orden por separado.')

test_modified_kn()
test_shared_counts()
timing_rows=[]

def fit_score_orders(fit_texts,fit_labels,app_texts,level,orders,phase,fold):
    started=time.perf_counter()
    mapping=make_encoder(fit_texts,level)
    encoded_fit=[encode_text(text,level,mapping) for text in fit_texts]
    encoded_app=[encode_text(text,level,mapping) for text in app_texts]
    preprocessing=time.perf_counter()-started
    matrices={order:np.zeros((len(app_texts),6)) for order in orders}
    fit_seconds=0.0
    score_seconds={order:0.0 for order in orders}
    for class_index in range(6):
        fit_started=time.perf_counter()
        model=ModifiedKN(max(orders),set(mapping.values())).fit(sequence for sequence,target in zip(encoded_fit,fit_labels) if target==class_index)
        fit_seconds+=time.perf_counter()-fit_started
        for order in orders:
            score_started=time.perf_counter()
            matrices[order][:,class_index]=[model.log_probability(sequence,order) for sequence in encoded_app]
            score_seconds[order]+=time.perf_counter()-score_started
        model.close()
        del model
        gc.collect()
    for order in orders:
        timing_rows.append(dict(fase=phase,pliegue=fold,nivel=level,orden=order,ajuste_compartido_s=preprocessing+fit_seconds,puntuacion_s=score_seconds[order],tiempo_amortizado_s=score_seconds[order]+(preprocessing+fit_seconds)/len(orders)))
    print(phase,fold,level,'ordenes',orders,'segundos',round(time.perf_counter()-started,2),flush=True)
    return matrices


Pruebas MKN: tres descuentos, normalizacion, continuaciones, BOS/EOS, UNK y capitalizacion correctos.
Prueba de conteos compartidos: mismos log P que entrenar cada orden por separado.


In [4]:
cv_started=time.perf_counter()
oof_scores={configuration['id']:np.zeros((len(y_development),6)) for configuration in ALL_CONFIGURATIONS}
fold_id=np.full(len(y_development),-1,dtype=int)
for fold_number,(fit_idx,app_idx) in enumerate(folds,1):
    assert not {text.lower() for text in texts_development[fit_idx]}&{text.lower() for text in texts_development[app_idx]}
    fit_labels=y_development[fit_idx]
    characters=fit_score_orders(texts_development[fit_idx],fit_labels,texts_development[app_idx],'char',CHAR_ORDERS,'CV',fold_number)
    words=fit_score_orders(texts_development[fit_idx],fit_labels,texts_development[app_idx],'word',WORD_ORDERS,'CV',fold_number)
    prior=np.log(np.bincount(fit_labels,minlength=6)/len(fit_labels))
    for configuration in ALL_CONFIGURATIONS:
        weight=configuration['peso']
        oof_scores[configuration['id']][app_idx]=weight*characters[configuration['car']]+(1-weight)*words[configuration['pal']]+prior
    fold_id[app_idx]=fold_number
    del characters,words
    gc.collect()
assert (fold_id>0).all()
oof_predictions={identifier:scores.argmax(axis=1) for identifier,scores in oof_scores.items()}
fold_f1={identifier:np.array([f1_score(y_development[app_idx],predictions[app_idx],labels=np.arange(6),average='macro',zero_division=0) for _,app_idx in folds]) for identifier,predictions in oof_predictions.items()}
result_rows,paired_rows=[],[]
for configuration in ALL_CONFIGURATIONS:
    identifier=configuration['id']
    differences=fold_f1[identifier]-fold_f1['REF']
    result_rows.append(dict(**configuration,f1_macro_oof=f1_score(y_development,oof_predictions[identifier],labels=np.arange(6),average='macro',zero_division=0),f1_pliegues_media=float(fold_f1[identifier].mean()),f1_pliegues_std=float(fold_f1[identifier].std(ddof=1)),diferencia_pareada_media=float(differences.mean()),diferencia_pareada_std=float(differences.std(ddof=1))))
    for number,(value,difference) in enumerate(zip(fold_f1[identifier],differences),1):
        paired_rows.append(dict(configuracion=identifier,pliegue=number,f1_macro=value,diferencia_con_referencia=difference))
results=pd.DataFrame(result_rows)
paired_table=pd.DataFrame(paired_rows)
print('TABLA COMPLETA: 8 CONFIGURACIONES Y REFERENCIA RECALCULADA; ddof=1')
print(results.to_string(index=False))
print('Referencia anterior publicada: 0.440081; recalculada:',float(results.loc[results.id=='REF','f1_macro_oof'].iloc[0]),'; no se usa el valor publicado para elegir.')
display(paired_table)
display(pd.DataFrame(timing_rows))
print('CV segundos:',round(time.perf_counter()-cv_started,2))


CV 1 char ordenes [3, 4, 5] segundos 19.05


CV 1 word ordenes [2, 3, 4] segundos 14.6


CV 2 char ordenes [3, 4, 5] segundos 19.68


CV 2 word ordenes [2, 3, 4] segundos 14.26


CV 3 char ordenes [3, 4, 5] segundos 18.48


...

CV 5 word ordenes [2, 3, 4] segundos 13.57


TABLA COMPLETA: 8 CONFIGURACIONES Y REFERENCIA RECALCULADA; ddof=1
 id  orden_lista  car  pal  peso  f1_macro_oof  f1_pliegues_media  f1_pliegues_std  diferencia_pareada_media  diferencia_pareada_std
REF            0    4    3  0.75      0.440081           0.440079         0.005308                  0.000000                0.000000
 C1            1    3    2  0.60      0.424827           0.424830         0.006656                 -0.015249                0.002060
 C2            2    3    2  0.90      0.417123           0.417097         0.006119                 -0.022981                0.002565
 C3            3    3    4  0.60      0.430618           0.430649         0.007155                 -0.009430                0.003064
 C4            4    3    4  0.90      0.421381           0.421357         0.005583                 -0.018722                0.002939
 C5            5    5    2  0.60      0.434659           0.434694         0.003835                 -0.005384                0.002781
 C

,configuracion,pliegue,f1_macro,diferencia_con_referencia
0,REF,1,0.440843,0.000000
1,REF,2,0.432485,0.000000
2,REF,3,0.440697,0.000000
3,REF,4,0.447354,0.000000
4,REF,5,0.439014,0.000000
5,C1,1,0.428788,-0.012055
6,C1,2,0.415363,-0.017123
7,C1,3,0.424782,-0.015915
8,C1,4,0.432921,-0.014433
9,C1,5,0.422296,-0.016718


,fase,pliegue,nivel,orden,ajuste_compartido_s,puntuacion_s,tiempo_amortizado_s
0,CV,1,char,3,5.732860,1.865548,3.776501
1,CV,1,char,4,5.732860,3.661505,5.572459
2,CV,1,char,5,5.732860,7.401399,9.312352
3,CV,1,word,2,4.378653,1.423736,2.883287
4,CV,1,word,3,4.378653,3.216184,4.675734
5,CV,1,word,4,4.378653,5.150544,6.610095
6,CV,2,char,3,5.889763,1.948797,3.912051
7,CV,2,char,4,5.889763,3.966989,5.930243
8,CV,2,char,5,5.889763,7.502573,9.465827
9,CV,2,word,2,4.309851,1.396531,2.833148


CV segundos: 164.76


In [5]:
accepted=results.loc[(results.id!='REF')&(results.diferencia_pareada_media>0)&(results.diferencia_pareada_media>results.diferencia_pareada_std)].sort_values(['diferencia_pareada_media','orden_lista'],ascending=[False,True])
PASA=not accepted.empty
selected=accepted.iloc[0].to_dict() if PASA else None
print('PASA' if PASA else 'NO PASA')
print('Regla fijada: media de diferencia pareada >0 y >desviacion muestral pareada; desempate por orden de lista.')
print('Aceptadas:',accepted.id.tolist(),'| Elegida:',selected['id'] if PASA else 'ninguna')
if PASA:
    print('Parametros elegidos:',int(selected['car']),int(selected['pal']),float(selected['peso']))
else:
    print('NO PASA: holdout y CSV se omiten explicitamente; no se creara age_submission_v7_hiperparametros.csv.')
print('Elegir entre 8 configuraciones introduce optimismo de seleccion; esta es la ultima ronda de v7 sobre la familia de n-gramas. La ronda queda cerrada despues de esta corrida.')


NO PASA
Regla fijada: media de diferencia pareada >0 y >desviacion muestral pareada; desempate por orden de lista.
Aceptadas: [] | Elegida: ninguna
NO PASA: holdout y CSV se omiten explicitamente; no se creara age_submission_v7_hiperparametros.csv.
Elegir entre 8 configuraciones introduce optimismo de seleccion; esta es la ultima ronda de v7 sobre la familia de n-gramas. La ronda queda cerrada despues de esta corrida.


In [6]:
if PASA:
    assert 'holdout_f1' not in globals(),'Holdout ya evaluado; no reutilizar.'
    started=time.perf_counter()
    selected_char,selected_word,selected_weight=int(selected['car']),int(selected['pal']),float(selected['peso'])
    texts_holdout=train.normalized.iloc[idx_holdout].to_numpy()
    assert not {text.lower() for text in texts_development}&{text.lower() for text in texts_holdout}
    characters=fit_score_orders(texts_development,y_development,texts_holdout,'char',[selected_char],'holdout',0)
    words=fit_score_orders(texts_development,y_development,texts_holdout,'word',[selected_word],'holdout',0)
    prior=np.log(np.bincount(y_development,minlength=6)/len(y_development))
    holdout_pred=(selected_weight*characters[selected_char]+(1-selected_weight)*words[selected_word]+prior).argmax(axis=1)
    y_holdout=y_all[idx_holdout]
    holdout_f1=f1_score(y_holdout,holdout_pred,labels=np.arange(6),average='macro',zero_division=0)
    print('F1 macro holdout v7_lower:',holdout_f1,'configuracion',selected['id'])
    report=classification_report(y_holdout,holdout_pred,labels=np.arange(6),target_names=CLASSES,output_dict=True,zero_division=0)
    display(pd.DataFrame(report).T.loc[CLASSES,['precision','recall','f1-score','support']])
    display(pd.DataFrame(confusion_matrix(y_holdout,holdout_pred,labels=np.arange(6)),index=CLASSES,columns=CLASSES))
    print('Holdout segundos:',round(time.perf_counter()-started,2))
    print('Evaluacion unica; no se cambia la seleccion tras observar holdout. No comparar directamente con particiones historicas distintas ni con Kaggle.')
    del characters,words
    gc.collect()
else:
    print('HOLDOUT OMITIDO: NO PASA; no se entrena ni evalua en holdout.')


HOLDOUT OMITIDO: NO PASA; no se entrena ni evalua en holdout.


In [7]:
if PASA:
    verify_csv(DATA_DIR/'age_test.csv',['id','text'],';',True)
    test=pd.read_csv(DATA_DIR/'age_test.csv',sep=';',encoding='utf-8-sig',dtype=str,keep_default_na=False)
    assert list(test.columns)==['id','text'] and test.id.is_unique and test.id.ne('').all()
    test_texts=test.text.fillna('').str.strip().map(html.unescape).map(normalize_text).to_numpy()
    characters=fit_score_orders(train.normalized.to_numpy(),y_all,test_texts,'char',[selected_char],'final',0)
    words=fit_score_orders(train.normalized.to_numpy(),y_all,test_texts,'word',[selected_word],'final',0)
    prior=np.log(np.bincount(y_all,minlength=6)/len(y_all))
    predictions=(selected_weight*characters[selected_char]+(1-selected_weight)*words[selected_word]+prior).argmax(axis=1)
    submission=pd.DataFrame({'id':test.id,'age_range':CLASSES[predictions]})
    assert list(submission.columns)==reference_columns==['id','age_range']
    assert len(submission)==len(test)==112892 and submission.id.is_unique and submission.id.tolist()==test.id.tolist()
    assert submission.age_range.isin(CLASSES).all() and submission.age_range.ne('').all()
    with SUB_PATH.open('x',encoding='utf-8',newline='') as handle:
        submission.to_csv(handle,index=False,sep=',')
    with SUB_PATH.open('rb') as handle: prefix=handle.read(64)
    assert not prefix.startswith(b'\xef\xbb\xbf') and prefix.splitlines()[0]==b'id,age_range'
    print('CSV VERIFICADO:',SUB_PATH.name,'112892 filas; IDs unicos y en orden; rangos validos; coma y sin BOM.')
    del characters,words
    gc.collect()
else:
    assert not SUB_PATH.exists()
    print('CSV OMITIDO: NO PASA; no se carga el test para predecir ni se genera salida.')


CSV OMITIDO: NO PASA; no se carga el test para predecir ni se genera salida.


In [8]:
display(results)
display(paired_table)
reference_measured=float(results.loc[results.id=='REF','f1_macro_oof'].iloc[0])
summary_text=f"""## Resumen final: cierre de la familia v7 de n-gramas

Resultado: **{'PASA' if PASA else 'NO PASA'}**. Referencia (4,3,0.75) recalculada: **{reference_measured:.6f}**, sin calibracion. Se evaluaron las ocho configuraciones fijadas con la misma normalizacion A, el mismo ModifiedKN y los cinco pliegues agrupados de v7_lower.

Configuracion elegida: **{selected['id'] if PASA else 'ninguna'}**. {'Holdout evaluado una sola vez y CSV generado con escritura exclusiva.' if PASA else 'Holdout y CSV omitidos explicitamente por la regla.'}

La regla exige diferencia pareada media positiva y mayor que su desviacion estandar muestral, ddof=1; no es una prueba formal de significacion. Elegir entre **8 configuraciones introduce optimismo de seleccion**. No se utiliza el holdout para elegir. **Esta es la ultima ronda de v7 sobre la familia de n-gramas y se cierra despues de esta corrida.**

El NPZ de v7_lower se reutilizo sin regenerar, con hashes del archivo, train y grupos verificados; cero fuga en minusculas. El holdout de v6 pudo tener coincidencias en minusculas y no se compara directamente con este. Los scores publicos de Kaggle tampoco son comparables directamente con holdout. Referencias historicas permitidas: v5 calibrado=0.43847, v6 sin calibrar=0.44106 y v6 calibrado=0.44353; no se usaron para elegir.

No se modificaron archivos existentes ni se instalaron paquetes. Sin commit, push ni envios a Kaggle. Salidas relativas y ejecucion con el entorno existente .venv-v6.
"""
display(Markdown(summary_text))
assert INPUT_HASHES=={name:hashlib.sha256((DATA_DIR/name).read_bytes()).hexdigest() for name in INPUT_HASHES}
assert PROTECTED_HASHES=={name:hashlib.sha256((OUT_DIR/name).read_bytes()).hexdigest() for name in PROTECTED_HASHES}
print('HASHES DE TODOS LOS ARCHIVOS PREEXISTENTES PROTEGIDOS: SIN CAMBIOS.')


,id,orden_lista,car,pal,peso,f1_macro_oof,f1_pliegues_media,f1_pliegues_std,diferencia_pareada_media,diferencia_pareada_std
0,REF,0,4,3,0.75,0.440081,0.440079,0.005308,0.000000,0.000000
1,C1,1,3,2,0.60,0.424827,0.424830,0.006656,-0.015249,0.002060
2,C2,2,3,2,0.90,0.417123,0.417097,0.006119,-0.022981,0.002565
3,C3,3,3,4,0.60,0.430618,0.430649,0.007155,-0.009430,0.003064
4,C4,4,3,4,0.90,0.421381,0.421357,0.005583,-0.018722,0.002939
5,C5,5,5,2,0.60,0.434659,0.434694,0.003835,-0.005384,0.002781
6,C6,6,5,2,0.90,0.435721,0.435723,0.003307,-0.004355,0.002642
7,C7,7,5,4,0.60,0.437677,0.437712,0.004485,-0.002367,0.001548
8,C8,8,5,4,0.90,0.437125,0.437116,0.003328,-0.002963,0.002969


,configuracion,pliegue,f1_macro,diferencia_con_referencia
0,REF,1,0.440843,0.000000
1,REF,2,0.432485,0.000000
2,REF,3,0.440697,0.000000
3,REF,4,0.447354,0.000000
4,REF,5,0.439014,0.000000
5,C1,1,0.428788,-0.012055
6,C1,2,0.415363,-0.017123
7,C1,3,0.424782,-0.015915
8,C1,4,0.432921,-0.014433
9,C1,5,0.422296,-0.016718


## Resumen final: cierre de la familia v7 de n-gramas

Resultado: **NO PASA**. Referencia (4,3,0.75) recalculada: **0.440081**, sin calibracion. Se evaluaron las ocho configuraciones fijadas con la misma normalizacion A, el mismo ModifiedKN y los cinco pliegues agrupados de v7_lower.

Configuracion elegida: **ninguna**. Holdout y CSV omitidos explicitamente por la regla.

La regla exige diferencia pareada media positiva y mayor que su desviacion estandar muestral, ddof=1; no es una prueba formal de significacion. Elegir entre **8 configuraciones introduce optimismo de seleccion**. No se utiliza el holdout para elegir. **Esta es la ultima ronda de v7 sobre la familia de n-gramas y se cierra despues de esta corrida.**

El NPZ de v7_lower se reutilizo sin regenerar, con hashes del archivo, train y grupos verificados; cero fuga en minusculas. El holdout de v6 pudo tener coincidencias en minusculas y no se compara directamente con este. Los scores publicos de Kaggle tampoco son comparables directamente con holdout. Referencias historicas permitidas: v5 calibrado=0.43847, v6 sin calibrar=0.44106 y v6 calibrado=0.44353; no se usaron para elegir.

No se modificaron archivos existentes ni se instalaron paquetes. Sin commit, push ni envios a Kaggle. Salidas relativas y ejecucion con el entorno existente .venv-v6.


HASHES DE TODOS LOS ARCHIVOS PREEXISTENTES PROTEGIDOS: SIN CAMBIOS.
